In [0]:
!pip install openai presidio-analyzer presidio-anonymizer Spacy
dbutils.library.restartPython()

In [0]:
import spacy
spacy.cli.download("en_core_web_lg")
nlp = spacy.load("en_core_web_lg")
dbutils.library.restartPython()

In [0]:
from presidio_analyzer import AnalyzerEngine, RecognizerRegistry, PatternRecognizer, Pattern
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig
import argparse
import json
from typing import List, Dict, Optional




In [0]:
class PresidioPIIMasker:
    def __init__(self):
        # Initialize the analyzer with default recognizers
        self.registry = RecognizerRegistry()
        self.registry.load_predefined_recognizers()
       
        # Add custom recognizers for specific PII types
       
        # Canadian SIN recognizer
        sin_pattern = Pattern(
            name="sin_pattern",
            regex=r"\b\d{3}[-\s]?\d{3}[-\s]?\d{3}\b",
            score=0.85
        )
        sin_recognizer = PatternRecognizer(
            supported_entity="CA_SIN",
            patterns=[sin_pattern],
            context=["sin", "social insurance", "insurance number", "verification"]
        )
        self.registry.add_recognizer(sin_recognizer)
       
        # Credit Card recognizer (enhanced)
        cc_pattern = Pattern(
            name="credit_card_pattern",
            regex=r"\b(?:\d{4}[-\s]?){3,4}\d{1,4}\b",
            score=0.9
        )
        cc_recognizer = PatternRecognizer(
            supported_entity="CREDIT_CARD",
            patterns=[cc_pattern],
            context=["card", "credit", "visa", "mastercard", "amex", "number"]
        )
        self.registry.add_recognizer(cc_recognizer)
       
        # CVV recognizer
        cvv_pattern = Pattern(
            name="cvv_pattern",
            regex=r"\bCVV\s*\d{3,4}\b",
            score=0.95
        )
        cvv_recognizer = PatternRecognizer(
            supported_entity="CVV",
            patterns=[cvv_pattern],
            context=["cvv", "security code", "card verification"]
        )
        self.registry.add_recognizer(cvv_recognizer)
       
        # Driver's License recognizer
        dl_pattern = Pattern(
            name="drivers_license_pattern",
            regex=r"\b[A-Za-z]\d{8,9}\b",
            score=0.7
        )
        dl_recognizer = PatternRecognizer(
            supported_entity="DRIVERS_LICENSE",
            patterns=[dl_pattern],
            context=["driver", "license", "dl", "id number"]
        )
        self.registry.add_recognizer(dl_recognizer)
       
        # Expiry date recognizer
        expiry_pattern = Pattern(
            name="expiry_pattern",
            regex=r"\b(?:expiry|expires?|exp|valid until)\s*(?:date)?[:\\s]*\d{1,2}[\\/\\-]\\d{2,4}\b",
            score=0.8
        )
        expiry_recognizer = PatternRecognizer(
            supported_entity="EXPIRY_DATE",
            patterns=[expiry_pattern],
            context=["expiry", "expires", "valid", "expiration"]
        )
        self.registry.add_recognizer(expiry_recognizer)
       
        # Initialize analyzer and anonymizer
        self.analyzer = AnalyzerEngine(registry=self.registry)
        self.anonymizer = AnonymizerEngine()
       
        # Define anonymization operators
        self.operators = {
            "PERSON": OperatorConfig("replace", {"new_value": "[NAME]"}),
            "PHONE_NUMBER": OperatorConfig("replace", {"new_value": "[PHONE]"}),
            "EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "[EMAIL]"}),
            "CREDIT_CARD": OperatorConfig("replace", {"new_value": "[CREDIT_CARD]"}),
            "CA_SIN": OperatorConfig("replace", {"new_value": "[SIN]"}),
            "CVV": OperatorConfig("replace", {"new_value": "[CVV]"}),
            "DRIVERS_LICENSE": OperatorConfig("replace", {"new_value": "[DRIVERS_LICENSE]"}),
            "EXPIRY_DATE": OperatorConfig("replace", {"new_value": "[EXPIRY_DATE]"}),
            "DATE_TIME": OperatorConfig("replace", {"new_value": "[DATE]"}),
            "LOCATION": OperatorConfig("replace", {"new_value": "[ADDRESS]"}),
            "NRP": OperatorConfig("replace", {"new_value": "[NRP]"}),  # Nationality/Religion/Political
            "IP_ADDRESS": OperatorConfig("replace", {"new_value": "[IP]"}),
            "US_SSN": OperatorConfig("replace", {"new_value": "[SSN]"}),
            "US_ITIN": OperatorConfig("replace", {"new_value": "[ITIN]"}),
            "US_PASSPORT": OperatorConfig("replace", {"new_value": "[PASSPORT]"}),
            "US_BANK_NUMBER": OperatorConfig("replace", {"new_value": "[BANK_ACCOUNT]"}),
            "IBAN_CODE": OperatorConfig("replace", {"new_value": "[IBAN]"}),
            "CRYPTO": OperatorConfig("replace", {"new_value": "[CRYPTO]"}),
            "UK_NHS": OperatorConfig("replace", {"new_value": "[NHS]"}),
            "ES_NIF": OperatorConfig("replace", {"new_value": "[NIF]"}),
            "IT_FISCAL_CODE": OperatorConfig("replace", {"new_value": "[FISCAL_CODE]"}),
            "IT_DRIVER_LICENSE": OperatorConfig("replace", {"new_value": "[DRIVERS_LICENSE]"}),
            "IT_VAT_CODE": OperatorConfig("replace", {"new_value": "[VAT]"}),
            "IT_PASSPORT": OperatorConfig("replace", {"new_value": "[PASSPORT]"}),
            "IT_IDENTITY_CARD": OperatorConfig("replace", {"new_value": "[ID_CARD]"}),
            "SG_NRIC_FIN": OperatorConfig("replace", {"new_value": "[NRIC]"}),
            "AU_ABN": OperatorConfig("replace", {"new_value": "[ABN]"}),
            "AU_ACN": OperatorConfig("replace", {"new_value": "[ACN]"}),
            "AU_TFN": OperatorConfig("replace", {"new_value": "[TFN]"}),
            "AU_MEDICARE": OperatorConfig("replace", {"new_value": "[MEDICARE]"}),
            "IN_PAN": OperatorConfig("replace", {"new_value": "[PAN]"}),
            "IN_AADHAAR": OperatorConfig("replace", {"new_value": "[AADHAAR]"}),
            "IN_VOTER": OperatorConfig("replace", {"new_value": "[VOTER_ID]"}),
            "IN_PASSPORT": OperatorConfig("replace", {"new_value": "[PASSPORT]"}),
            "FI_PERSONAL_IDENTITY_CODE": OperatorConfig("replace", {"new_value": "[PERSONAL_ID]"}),
        }
   
    def analyze(self, text: str, language: str = "en") -> List[Dict]:
        """
        Analyze text and detect PII entities
        """
        results = self.analyzer.analyze(
            text=text,
            language=language,
            entities=None,  # All entities
            return_decision_process=True
        )
        return results
   
    def anonymize(self, text: str, analyzer_results: List, operators: Optional[Dict] = None) -> Dict:
        """
        Anonymize detected PII entities
        """
        if operators is None:
            operators = self.operators
           
        result = self.anonymizer.anonymize(
            text=text,
            analyzer_results=analyzer_results,
            operators=operators
        )
        return result
   
    def mask_transcript(self, text: str, language: str = "en") -> Dict:
        """
        Complete pipeline: analyze and anonymize
        """
        # Analyze for PII
        analyzer_results = self.analyze(text, language)
       
        # Anonymize
        anonymized_result = self.anonymize(text, analyzer_results)
       
        # Create detailed report
        report = {
            "original_text": text,
            "masked_text": anonymized_result.text,
            "entities_detected": []
        }
       
        for result in analyzer_results:
            entity_info = {
                "entity_type": result.entity_type,
                "start": result.start,
                "end": result.end,
                "score": result.score,
                "original_value": text[result.start:result.end]
            }
            report["entities_detected"].append(entity_info)
       
        return report
   
    def generate_summary(self, report: Dict) -> str:
        """
        Generate a human-readable summary of the masking operation
        """
        lines = [
            "PII Masking Summary (Presidio)",
            "=" * 60,
            "",
            f"Total entities detected: {len(report['entities_detected'])}",
            ""
        ]
       
        # Group by entity type
        from collections import defaultdict
        by_type = defaultdict(list)
       
        for entity in report['entities_detected']:
            by_type[entity['entity_type']].append(entity)
       
        for entity_type, entities in sorted(by_type.items()):
            lines.append(f"{entity_type}: {len(entities)} instance(s)")
            for entity in entities:
                lines.append(f"  - '{entity['original_value']}' (confidence: {entity['score']:.2f})")
            lines.append("")
       
        return "\n".join(lines)
 
 


In [0]:
Text = """
Agent: Thank you for calling SecureBank Credit Card Services. This is Michael Thompson speaking. May I have your name please?
 
Customer: Yes, hi. My name is Jennifer Sarah Williams.
 
Agent: Thank you, Ms. Williams. For security purposes, may I have your date of birth?
 
Customer: Sure, it's March 15, 1985. I'm 39 years old.
 
Agent: And could you please verify your phone number on file?
 
Customer: Yes, my cell is 416-555-7890 and my home number is 647-555-1234.
 
Agent: Thank you. And your complete mailing address?
 
Customer: It's 1234 Maple Street, Apartment 502, Toronto, Ontario, M5V 3A8.
 
Agent: Perfect. Now, may I have your Social Insurance Number for verification?
 
Customer: Yes, it's 123-456-789.
 
Agent: Thank you for verifying. I see you have the SecureBank Platinum Rewards Card ending in 4521. Are you calling about a transaction on this card?
 
Customer: Yes, I noticed a charge of $1,247.99 at Electronics World on May 15th that I didn't make. My card number is 4532-1234-5678-9012 with expiry 09/26 and CVV 123.
 
Agent: I understand your concern. Let me check that transaction. While I pull up the details, can you confirm your email address?
 
Customer: It's jennifer.williams1985@email.com.
 
Agent: Thank you. I can see the transaction you're referring to. For your protection, I'll block this card immediately and issue a replacement. Your new card will be sent to 1234 Maple Street within 5-7 business days.
 
Customer: Great, thank you. My driver's license number is A123456789 if you need additional ID.
 
Agent: That's helpful, thank you. Is there anything else I can assist you with today?
 
Customer: No, that's all. Thanks for your help, Michael.
 
Agent: You're welcome, Ms. Williams. Have a great day! 
"""

In [0]:
Text

In [0]:

masker = PresidioPIIMasker()

# Use the mask_transcript method to analyze and anonymize the text
report = masker.mask_transcript(Text)

In [0]:
report